<span style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">An Exception was encountered at '<a href="#papermill-error-cell">In [1]</a>'.</span>

<span id="papermill-error-cell" style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">Execution using papermill encountered an exception here and stopped:</span>

In [1]:




import pennylane as qml
import pennylane.numpy as np

def U(input_list):
    """Implements a quantum operation that computes the parity of the number of forests."""
    parity_qubit = 8
    temp_qubit = 7

    # Reset the parity qubit
    qml.Reset(wires=parity_qubit)

    # Encode input into qubits 0-7
    for i in range(8):
        if input_list[i] == 1:
            qml.X(wires=i)

    # Check if position 0 is a tree (start of a potential forest)
    qml.CRY(np.pi, wires=[0, parity_qubit])

    # For each pair of consecutive positions, detect 0 -> 1 transition
    for i in range(1, 8):
        # Flip i-1 to detect if it was 0
        qml.PauliX(wires=i - 1)
        qml.CCX(wires=[i - 1, i, temp_qubit])
        qml.PauliX(wires=i - 1)

        # Use temp_qubit to flip parity_qubit
        qml.CRY(np.pi, wires=[temp_qubit, parity_qubit])

        # Reset temp_qubit for next iteration
        qml.Reset(wires=temp_qubit)


def run_forest_parity(input_list):
    """Runs the quantum circuit and returns the parity of the number of forests."""
    dev = qml.device("default.qubit", wires=9)

    @qml.qnode(dev)
    def circuit(input_list):
        U(input_list)
        return qml.expval(qml.PauliZ(wires=8))

    expectation = circuit(input_list)
    # Map the expectation value (in [-1, 1]) to 0 or 1
    return 1 if expectation < 0 else 0


# Test cases
test_cases = [
    ([1, 0, 1, 1, 0, 1, 1, 1], 1),  # Expected: 1 (3 forests)
    ([0, 0, 0, 0, 0, 1, 0, 1], 0),  # Expected: 0 (2 forests)
    ([1, 1, 0, 1, 0, 0, 1, 0], 1),  # Expected: 1 (3 forests)
    ([0, 0, 0, 0, 0, 0, 0, 0], 0),  # Expected: 0 (0 forests)
    ([1, 1, 1, 1, 1, 1, 1, 1], 1),  # Expected: 1 (1 forest)
    ([0, 1, 1, 0, 1, 1, 1, 0], 0),  # Expected: 0 (2 forests)
]

# Run tests
for idx, (input_list, expected) in enumerate(test_cases):
    result = run_forest_parity(input_list)
    print(f"Test case {idx + 1}: {'Passed' if result == expected else 'Failed'} (Got {result}, Expected {expected})")

SyntaxError: invalid syntax (275523626.py, line 1)